# Часть 1. Знакомство с данными в PostgreSQL


## Загрузим и изучим данные

## Подключение и загрузка таблиц

Импортируем необходимые библиотеки:

In [1]:
!pip install boto3 python-dotenv

import os
import boto3
import importlib.util
import numpy as n
import psycopg2
import pandas as pd
from dotenv import load_dotenv

# Читает файл .env из корня репозитория (рядом с тетрадкой)
# и раскладывает его содержимое в os.environ
load_dotenv(override=True)

# Настройка отображения:
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.precision', 2)

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Подключимся к базе данных:

In [2]:
# Параметры подключения из сниппета платформы.
# ВАЖНО: здесь это допустимо (локальная разведка в тетрадке),
# но в репозиторий такие секреты не пойдут — там будет Airflow Connection.
CONN_PARAMS = {
    "dbname": os.environ["DB_NAME"],
    "host": os.environ["DB_HOST"],
    "port": int(os.environ.get("DB_PORT", "6432")),
    "user": os.environ["DB_USER"],
    "password": os.environ["DB_PASSWORD"],
}
conn = psycopg2.connect(**CONN_PARAMS)


TABLES = ["customers", "sessions", "events", "orders"]
data = {}
for table in TABLES:
    data[table] = pd.read_sql(f"SELECT * FROM public.{table}", conn)
conn.close()

customers, sessions, events, orders = (data[name] for name in TABLES)

# Размер, типы колонок и пример строк каждой таблицы
for name in TABLES:
    df = data[name]
    print(f"\n=== ТАБЛИЦА: {name} (строк: {df.shape[0]}, колонок: {df.shape[1]}) ===")
    print("-" * 50)
    display(df.dtypes.to_frame(name='Тип данных'))
    display(df.head(3))

C:\Users\svasl\AppData\Local\Temp\ipykernel_6156\2786386671.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data[table] = pd.read_sql(f"SELECT * FROM public.{table}", conn)



=== ТАБЛИЦА: customers (строк: 17846, колонок: 7) ===
--------------------------------------------------


,Тип данных
customer_id,int64
name,object
email,object
country,object
age,int64
signup_date,object
marketing_opt_in,bool


,customer_id,name,email,country,age,signup_date,marketing_opt_in
0,1,Jennifer Salinas,nicholas59@example.org,JP,71,2020-09-04,True
1,2,Phillip Ramos,christinarubio@example.com,IN,26,2020-04-05,False
2,3,Dawn Fowler,jessica03@example.org,BR,21,2023-08-31,True



=== ТАБЛИЦА: sessions (строк: 37699, колонок: 6) ===
--------------------------------------------------


...

,Тип данных
event_id,int64
session_id,int64
timestamp,datetime64[ns]
event_type,object
product_id,float64
qty,float64
cart_size,float64
payment,object
discount_pct,float64
amount_usd,float64


,event_id,session_id,timestamp,event_type,product_id,qty,cart_size,payment,discount_pct,amount_usd
0,11,2,2025-01-31 21:48:42,page_view,929.0,NaN,NaN,None,NaN,NaN
1,12,2,2025-01-31 21:57:42,page_view,226.0,NaN,NaN,None,NaN,NaN
2,13,2,2025-01-31 21:59:07,add_to_cart,226.0,1.0,NaN,None,NaN,NaN



=== ТАБЛИЦА: orders (строк: 10652, колонок: 10) ===
--------------------------------------------------


,Тип данных
order_id,int64
customer_id,int64
order_time,datetime64[ns]
payment_method,object
discount_pct,float64
subtotal_usd,float64
total_usd,float64
country,object
device,object
source,object


,order_id,customer_id,order_time,payment_method,discount_pct,subtotal_usd,total_usd,country,device,source
0,1,13917,2025-01-31 23:07:42,card,20.0,107.15,85.72,PL,desktop,organic
1,2,1022,2024-02-19 01:17:50,card,0.0,116.17,116.17,FR,tablet,organic
2,3,6145,2024-12-04 20:24:13,card,0.0,137.35,137.35,US,mobile,organic


**Комментарий:**

1. `customers.signup_date` имеет тип `object` (строка). Её обязательно нужно будет перевести в datetime при предобработке, чтобы можно было корректно сравнивать с `run_date`.
2. Временные поля в `sessions`, `events` и `orders` уже подгрузились как `datetime64[ns]`. Это хорошо, но нам всё равно нужно будет проверить их на наличие невалидных дат `(NaT)` и, возможно, привести к единому часовому поясу `(UTC)`, чтобы избежать скрытых багов при фильтрации `timestamp < run_date.`
3. В таблице `events` есть пропуски `(NaN)` в числовых полях (product_id, qty и т.д.), а в поле payment встречается None. Для признака "количество уникальных товаров" мы будем игнорировать строки, где product_id — пустое значение.
4. Количество строк идеально совпадает с описанием (17 846 / 37 699 / 239 089 / 10 652).

## Ключи, дубликаты, границы дат, пропуски

In [3]:
# # 1) Уникальность ключей
print(f"customers: rows={len(customers)}, unique customer_id={customers['customer_id'].nunique()}")
print(f"sessions:  rows={len(sessions)}, unique session_id={sessions['session_id'].nunique()}, unique customer_id={sessions['customer_id'].nunique()}")
print(f"events:    rows={len(events)}, unique event_id={events['event_id'].nunique()}, unique session_id={events['session_id'].nunique()}")
print(f"orders:    rows={len(orders)}, unique order_id={orders['order_id'].nunique()}, unique customer_id={orders['customer_id'].nunique()}")

print("\n--- Дубликаты по ключам ---")
print(f"customers by customer_id: {customers['customer_id'].duplicated().sum()}")
print(f"sessions  by session_id:  {sessions['session_id'].duplicated().sum()}")
print(f"events    by event_id:    {events['event_id'].duplicated().sum()}")
print(f"orders    by order_id:    {orders['order_id'].duplicated().sum()}")

# # 2) Границы дат
date_columns = {
    "customers": "signup_date",
    "sessions": "start_time",
    "events": "timestamp",
    "orders": "order_time",
}
print("\n--- Границы дат ---")
for name, col in date_columns.items():
    parsed = pd.to_datetime(data[name][col], errors="coerce")
    print(f"{name}.{col}: min={parsed.min()}, max={parsed.max()}, невалидных={parsed.isna().sum()}")

# # 3) Пропуски в полях
important_columns = {
    "customers": ["customer_id", "signup_date"],
    "sessions": ["session_id", "customer_id", "start_time"],
    "events": ["event_id", "session_id", "timestamp", "event_type", "product_id"],
    "orders": ["order_id", "customer_id", "order_time", "total_usd"],
}
print("\n--- Пропуски в важных полях ---")
for name, cols in important_columns.items():
    print(f"\n[{name}]")
    # Превращаем в красивую HTML-таблицу для вывода на экран
    display(data[name][cols].isna().sum().to_frame(name='Кол-во пропусков'))

customers: rows=17846, unique customer_id=17846
sessions:  rows=37699, unique session_id=37699, unique customer_id=16894
events:    rows=239089, unique event_id=239089, unique session_id=37699
orders:    rows=10652, unique order_id=10652, unique customer_id=8230

--- Дубликаты по ключам ---
customers by customer_id: 0
sessions  by session_id:  0
events    by event_id:    0
orders    by order_id:    0

--- Границы дат ---
customers.signup_date: min=2020-01-01 00:00:00, max=2025-10-31 00:00:00, невалидных=0
sessions.start_time: min=2024-01-01 00:57:40, max=2025-10-31 23:34:11, невалидных=0
events.timestamp: min=2024-01-01 01:02:40, max=2025-11-01 01:50:04, невалидных=0
orders.order_time: min=2024-01-01 07:08:06, max=2025-10-31 22:59:41, невалидных=0

--- Пропуски в важных полях ---

[customers]


,Кол-во пропусков
customer_id,0
signup_date,0



[sessions]


,Кол-во пропусков
session_id,0
customer_id,0
start_time,0



[events]


,Кол-во пропусков
event_id,0
session_id,0
timestamp,0
event_type,0
product_id,24881



[orders]


,Кол-во пропусков
order_id,0
customer_id,0
order_time,0
total_usd,0


**Комментарий:**

1. Дубликатов по первичным ключам нет. В `events` ровно **239 089** уникальных `event_id`, в `sessions` — **37 699** `session_id`, в `orders` — **10 652** `order_id`. Дедупликация через `.drop_duplicates(`) не изменит размер таблиц, но в итоговом пайплайне (и в .py-модуле) мы её всё равно оставим как защитную меру от «грязи» в проде.
2. Не все пользователи активны. У нас **17 846** пользователей в `customers`, но сессии были только у **16 894**, а покупки совершали **8 230**.
*Что это значит для нас:* Итоговая таблица признаков должна строиться от справочника `customers` (чтобы сохранить всех пользователей), а таблицы активностей будем джойнить через `left join`. Тем, у кого не было событий/сессий/заказов в окне, признаки заполним нулями (как и требуется в ТЗ).

3. Границы дат идеальны. Период с января 2024 по конец октября/начало ноября 2025. Невалидных дат нет. Значит, для тестового `run_date` мы можем взять, например, `2025-10-01` (чтобы гарантированно иметь `30 дней` истории до неё и данные после неё).
4. Типы дат. `signup_date` — это `object`, его обязательно нужно парсить через `pd.to_datetime`. Остальные временные колонки уже подгрузились как `datetime64[ns]`.
5. Пропуски в `product_id`. В `events` **24 881** строка без `product_id`. Это ожидаемо (например, пользователь просто зашёл на главную или в корзину, а не на карточку товара). При расчёте признака «Количество уникальных `product_id`» мы просто отфильтруем такие строки `(df['product_id'].notna())`.

## Связи таблиц и воронка покупки

In [4]:
# 4)Referential integrity: насколько полно таблицы стыкуются по ключам.
# Это определяет, какие join будут безопасны (inner не потеряет данные или потеряет предсказуемо).
print("events.session_id НЕ найден в sessions:   ",
      (~events["session_id"].isin(set(sessions["session_id"]))).sum())
print("sessions.customer_id НЕ найден в customers:",
      (~sessions["customer_id"].isin(set(customers["customer_id"]))).sum())
print("orders.customer_id НЕ найден в customers:  ",
      (~orders["customer_id"].isin(set(customers["customer_id"]))).sum())

# 5) Воронка page_view -> add_to_cart -> checkout -> purchase
funnel_stages = ["page_view", "add_to_cart", "checkout", "purchase"]
funnel = (
    events.groupby("event_type")
    .agg(events=("event_id", "size"), unique_sessions=("session_id", "nunique"))
    .reindex(funnel_stages)
)
funnel["conv_vs_prev"] = funnel["events"] / funnel["events"].shift(1)
print("\nВоронка покупки:")
display(funnel)

# Есть ли типы событий вне воронки и пропуски в event_type
print("\nВсе значения event_type:")
display(events["event_type"].value_counts(dropna=False))

events.session_id НЕ найден в sessions:    0
sessions.customer_id НЕ найден в customers: 0
orders.customer_id НЕ найден в customers:   0

Воронка покупки:


,events,unique_sessions,conv_vs_prev
event_type,,,
page_view,169300,37699,NaN
add_to_cart,44908,25601,0.27
checkout,14229,14229,0.32
purchase,10652,10652,0.75



Все значения event_type:


event_type
page_view      169300
add_to_cart     44908
checkout        14229
purchase        10652
Name: count, dtype: int64

**Комментарий:**

1. `Referential integrity` идеальная: ни одного «осиротевшего» `session_id` в `events`, ни одного потерянного `customer_id` в `sessions` и `orders`. Значит, `inner join events×sessions` безопасен (ничего не потеряем), а базу для итоговой таблицы всё равно берём из `customers` через `left join`, чтобы сохранить всех **17 846** пользователей.
2. Воронка согласована: `169 300 → 44 908 → 14 229 → 10 652` событий; конверсии `26.5% → 31.7% → 74.9%`. Важная деталь: число `purchase-событий` **(10 652)** точно равно числу строк в `orders`, и каждая покупка лежит в своей сессии (**10 652** уникальных сессий). То есть события и заказы не противоречат друг другу: конверсии считаем по `events`, деньги — по `orders`, как и требует ТЗ.
4. Типов событий ровно четыре, вне воронки ничего нет, пропусков в `event_type` нет — дополнительных фильтров не нужно.

**Промежуточный вывод:**

1. Размеры и ключи совпадают с описанием; дубликатов по PK нет — дедупликация оставлена как защитная операция.
2. Связи полные (0 сирот): `events→sessions→customers` и `orders→customers`. `JOIN events×sessions — inner`; итоговая таблица строится от всех `customers (left join)`, неактивным пользователям признаки заполняются `0 / −1`.
3. Период данных: `2024-01-01 … 2025-10-31` (`events` до 2025-11-01); `signup_date` с 2020 г. Невалидных дат нет; `signup_date` хранится строкой → парсим в `datetime`.
4. Активность: сессии есть у **16 894** из **17 846** клиентов, покупки — у**8 230** → ожидаем много нулевых признаков.
5. `product_id` пуст в **24 881** строке `events` (~10%) → фильтр `notna` применяется только внутри признака уникальных товаров, счётчики событий считаем по всем строкам.
6. Все временные колонки `naive` (без TZ) → приводим к` UTC`; `run_date` трактуем как полночь `UTC`; фильтр строгий: `time < run_date` (анти-утечка).
7. Демонстрационная дата среза: `run_date` = 2025-10-01 (есть ≥30 дней истории до и ≥7 дней данных после для будущей целевой переменной).

# Часть 2. Подготовка batch-признаков


## Подготовим данные

In [5]:
# ================= Задание 2.1: подготовка данных =================
RUN_DATE = "2025-10-01"                     # демонстрационная дата среза T
run_dt = pd.Timestamp(RUN_DATE, tz="UTC")   # все сравнения времени — относительно неё

def to_utc(series: pd.Series) -> pd.Series:
    """Единый тип (datetime64) и единый часовой пояс (UTC).
    Невалидные значения становятся NaT — такие строки отбросим дальше."""
    parsed = pd.to_datetime(series, errors="coerce")
    if parsed.dt.tz is None:
        parsed = parsed.dt.tz_localize("UTC")
    else:
        parsed = parsed.dt.tz_convert("UTC")
    return parsed

def preprocess_customers(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["signup_date"] = to_utc(out["signup_date"])   # строка -> datetime UTC
    out = out[out["signup_date"].notna()]             # отбрасываем невалидные даты
    return out.drop_duplicates(subset=["customer_id"])

def preprocess_sessions(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["start_time"] = to_utc(out["start_time"])
    out = out[out["start_time"].notna()]
    return out.drop_duplicates(subset=["session_id"])

def preprocess_events(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["timestamp"] = to_utc(out["timestamp"])
    out = out[out["timestamp"].notna()]
    return out.drop_duplicates(subset=["event_id"])

def preprocess_orders(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["order_time"] = to_utc(out["order_time"])
    out = out[out["order_time"].notna()]
    out = out.drop_duplicates(subset=["order_id"])
    # защита денежного признака: всё, что не приводится к числу, -> 0.0
    out["total_usd"] = pd.to_numeric(out["total_usd"], errors="coerce").fillna(0.0)
    return out

customers_pp = preprocess_customers(customers)
sessions_pp  = preprocess_sessions(sessions)
events_pp    = preprocess_events(events)
orders_pp    = preprocess_orders(orders)

# У событий нет собственного customer_id — присоединяем его через sessions.
# inner join безопасен: целостность связей проверена в задании 1.1 (0 сирот).
events_pp = events_pp.merge(sessions_pp[["session_id", "customer_id"]],
                            on="session_id", how="inner")

# Анти-утечка: единый срез «доступных данных» — строго раньше T.
# Дальше окна 7/30 дней будут браться только из этих кадров.
events_pp  = events_pp[events_pp["timestamp"]  < run_dt]
sessions_pp = sessions_pp[sessions_pp["start_time"] < run_dt]
orders_pp  = orders_pp[orders_pp["order_time"]  < run_dt]

print(f"run_date = {RUN_DATE}")
print("Строк после предобработки и среза по T:")
print(f"  customers: {len(customers_pp)} (сырых {len(customers)})")
print(f"  sessions:  {len(sessions_pp)} (сырых {len(sessions)})")
print(f"  events:    {len(events_pp)} (сырых {len(events)})")
print(f"  orders:    {len(orders_pp)} (сырых {len(orders)})")

print("\nПроверка единого типа и TZ:")
print("  signup_date:", customers_pp["signup_date"].dt.tz)
print("  start_time: ", sessions_pp["start_time"].dt.tz)
print("  timestamp:  ", events_pp["timestamp"].dt.tz)
print("  order_time: ", orders_pp["order_time"].dt.tz)

print("\nКонтроль анти-утечки (максимум времени в каждом кадре < run_date):")
print("  events max:  ", events_pp["timestamp"].max()  < run_dt)
print("  sessions max:", sessions_pp["start_time"].max() < run_dt)
print("  orders max:  ", orders_pp["order_time"].max()  < run_dt)

run_date = 2025-10-01
Строк после предобработки и среза по T:
  customers: 17846 (сырых 17846)
  sessions:  35959 (сырых 37699)
  events:    228042 (сырых 239089)
  orders:    10159 (сырых 10652)

Проверка единого типа и TZ:
  signup_date: UTC
  start_time:  UTC
  timestamp:   UTC
  order_time:  UTC

Контроль анти-утечки (максимум времени в каждом кадре < run_date):
  events max:   True
  sessions max: True
  orders max:   True


- `customers`: **17 846 → 17 846**. Справочник не фильтруется по времени — это база итоговой таблицы (одна строка на каждого клиента).
- `sessions`: **37 699 → 35 959 (−1 740)**: отсечены сессии, начавшиеся `2025-10-01` и позже.
- `events`: **239 089 → 228 042 (−11 047)**: отсечены события с `timestamp >= T`.
- `orders`: **10 652 → 10 159 (−493):** отсечены заказы с `order_time >= T.`
- Все четыре временные колонки в `UTC`, три проверки анти-утечки дали `True` — в расчёт попадут только данные, существовавшие до `T = 2025-10-01`.
Одно проектное решение фиксируем явно: множество `customer_id` во всех срезах одинаковое (все **17 846** клиентов), потому что требование ТЗ — «инферензный срез на тех же `customer_id`, что и обучающие». Справочник трактуем как статичный; фильтр по `signup_date` не применяем (его нет в правилах фильтрации ТЗ).

## Рассчитаем признаки

In [6]:
# ================= Задание 2.2: расчёт batch-признаков =================
WINDOW_7D  = pd.Timedelta(days=7)
WINDOW_30D = pd.Timedelta(days=30)
FUNNEL = ["page_view", "add_to_cart", "checkout", "purchase"]

# Итоговый состав и порядок колонок (фиксируем как контракт для модуля и DAG)
FEATURE_COLUMNS = [
    "customer_id", "run_date",
    "page_view_cnt_7d", "page_view_cnt_30d",
    "add_to_cart_cnt_7d", "add_to_cart_cnt_30d",
    "view_to_cart_conv_7d", "view_to_cart_conv_30d",
    "cart_to_purchase_conv_7d", "cart_to_purchase_conv_30d",
    "unique_products_7d", "unique_products_30d",
    "avg_session_duration_sec_30d",
    "sessions_cnt_7d", "sessions_cnt_30d",
    "days_since_last_purchase",
    "orders_cnt_30d", "orders_sum_usd_30d", "orders_avg_usd_30d",
]

def compute_batch_features(customers_df, sessions_df, events_df, orders_df, run_dt):
    """Признаки для пар (customer_id, run_date).

    На вход подаются УЖЕ предобработанные и обрезанные по `time < run_dt` кадры
    (задание 2.1), поэтому утечка из будущего исключена конструктивно.
    """
    # --- Длина сессий в секундах: max(timestamp) - min(timestamp) внутри сессии.
    #     У сессии нет событий в доступном срезе -> длина 0.0 (правило ТЗ).
    dur = (
        events_df.groupby("session_id")["timestamp"]
        .agg(lambda ts: (ts.max() - ts.min()).total_seconds())
        .rename("duration_sec")
        .reset_index()
    )
    session_duration = (
        sessions_df[["session_id"]]
        .merge(dur, on="session_id", how="left")
        .fillna({"duration_sec": 0.0})
    )

    def in_window(series, window):
        """Окно [run_dt - window, run_dt): строго раньше T и не старее окна."""
        return (series >= run_dt - window) & (series < run_dt)

    def event_features(window):
        w = events_df[in_window(events_df["timestamp"], window)]
        cnt = (
            w.pivot_table(index="customer_id", columns="event_type",
                          values="event_id", aggfunc="count")
            .reindex(columns=FUNNEL)          # гарантируем все 4 типа событий
            .fillna(0)
            .astype(int)
        )
        uniq = (
            w.loc[w["product_id"].notna()]    # только строки с заданным product_id
            .groupby("customer_id")["product_id"].nunique()
            .rename("unique_products")
        )
        block = cnt.join(uniq, how="left")
        block["unique_products"] = block["unique_products"].fillna(0).astype(int)
        suffix = f"{window.days}d"
        return block.rename(
            columns=lambda c: f"{c}_cnt_{suffix}" if c in FUNNEL else f"{c}_{suffix}"
        )

    def sessions_count(window):
        w = sessions_df[in_window(sessions_df["start_time"], window)]
        return (w.groupby("customer_id")["session_id"].count()
                .rename(f"sessions_cnt_{window.days}d"))

    ev_7, ev_30   = event_features(WINDOW_7D), event_features(WINDOW_30D)
    ses_7, ses_30 = sessions_count(WINDOW_7D), sessions_count(WINDOW_30D)

    # средняя длина сессий за 30 дней
    w30 = sessions_df[in_window(sessions_df["start_time"], WINDOW_30D)]
    avg_dur_30 = (
        w30.merge(session_duration, on="session_id", how="left")
        .groupby("customer_id")["duration_sec"].mean()
        .rename("avg_session_duration_sec_30d")
    )

    # денежные агрегаты — только из orders, окно 30 дней
    o30 = orders_df[in_window(orders_df["order_time"], WINDOW_30D)]
    ord_cnt = o30.groupby("customer_id")["order_id"].count().rename("orders_cnt_30d")
    ord_sum = o30.groupby("customer_id")["total_usd"].sum().rename("orders_sum_usd_30d")
    ord_avg = o30.groupby("customer_id")["total_usd"].mean().rename("orders_avg_usd_30d")

    # давность последней покупки по всей истории до T; не было заказов -> -1
    last_purchase = orders_df.groupby("customer_id")["order_time"].max()
    days_since = (run_dt - last_purchase).dt.days.rename("days_since_last_purchase")

    feats = pd.concat([ev_7, ev_30, ses_7, ses_30, avg_dur_30,
                       ord_cnt, ord_sum, ord_avg, days_since], axis=1)
    feats = feats.reindex(customers_df["customer_id"].unique())  # все клиенты, даже неактивные
    feats.index.name = "customer_id"
    feats = feats.reset_index()

    # --- Заполнение пропусков по правилам ТЗ
    int_cols = ["page_view_cnt_7d", "page_view_cnt_30d",
                "add_to_cart_cnt_7d", "add_to_cart_cnt_30d",
                "unique_products_7d", "unique_products_30d",
                "sessions_cnt_7d", "sessions_cnt_30d", "orders_cnt_30d"]
    float_cols = ["avg_session_duration_sec_30d", "orders_sum_usd_30d", "orders_avg_usd_30d"]
    feats[int_cols] = feats[int_cols].fillna(0).astype(int)          # счётчики -> 0
    feats[float_cols] = feats[float_cols].fillna(0.0).astype(float)  # вещественные -> 0.0
    feats["days_since_last_purchase"] = feats["days_since_last_purchase"].fillna(-1).astype(int)

    # --- Конверсии: деление на ноль -> 0.0
    def safe_ratio(num, den):
        return (num / den.where(den > 0)).fillna(0.0).astype(float)

    feats["view_to_cart_conv_7d"]      = safe_ratio(feats["add_to_cart_cnt_7d"],  feats["page_view_cnt_7d"])
    feats["view_to_cart_conv_30d"]     = safe_ratio(feats["add_to_cart_cnt_30d"], feats["page_view_cnt_30d"])
    feats["cart_to_purchase_conv_7d"]  = safe_ratio(feats["purchase_cnt_7d"],     feats["add_to_cart_cnt_7d"])
    feats["cart_to_purchase_conv_30d"] = safe_ratio(feats["purchase_cnt_30d"],    feats["add_to_cart_cnt_30d"])

    feats["run_date"] = run_dt.date().isoformat()
    return feats[FEATURE_COLUMNS]   # purchase_cnt_* остаются внутри функции и не выходят наружу

features = compute_batch_features(customers_pp, sessions_pp, events_pp, orders_pp, run_dt)

# ------------------------- Проверки результата -------------------------
checks = {
    "одна строка на (customer_id, run_date)": not features.duplicated(["customer_id", "run_date"]).any(),
    "строк = числу клиентов":                 len(features) == customers_pp["customer_id"].nunique(),
    "состав и порядок колонок = контракт":    list(features.columns) == FEATURE_COLUMNS,
    "нет NaN":                                not features.isna().any().any(),
    "счётчики 7d <= 30d":                     bool(
        (features["page_view_cnt_7d"] <= features["page_view_cnt_30d"]).all()
        and (features["add_to_cart_cnt_7d"] <= features["add_to_cart_cnt_30d"]).all()
        and (features["unique_products_7d"] <= features["unique_products_30d"]).all()
        and (features["sessions_cnt_7d"] <= features["sessions_cnt_30d"]).all()),
    "days_since: -1 или >= 0":                bool(((features["days_since_last_purchase"] == -1)
                                                    | (features["days_since_last_purchase"] >= 0)).all()),
    "конверсии >= 0":                         bool((features[[c for c in FEATURE_COLUMNS if "conv" in c]] >= 0).all().all()),
}
for name, ok in checks.items():
    print(("OK   | " if ok else "FAIL | ") + name)

display("\nАктивные пользователи (заказы за 30 дней):")
display(features[features["orders_cnt_30d"] > 0].head(5))
display("\nНеактивные пользователи (нет сессий за 30 дней):")
display(features[features["sessions_cnt_30d"] == 0].head(5))
display("\nСводная статистика:")
display(features.describe().T.round(2))

OK   | одна строка на (customer_id, run_date)
OK   | строк = числу клиентов
OK   | состав и порядок колонок = контракт
OK   | нет NaN
OK   | счётчики 7d <= 30d
OK   | days_since: -1 или >= 0
OK   | конверсии >= 0


'\nАктивные пользователи (заказы за 30 дней):'

,customer_id,run_date,page_view_cnt_7d,page_view_cnt_30d,add_to_cart_cnt_7d,add_to_cart_cnt_30d,view_to_cart_conv_7d,view_to_cart_conv_30d,cart_to_purchase_conv_7d,cart_to_purchase_conv_30d,unique_products_7d,unique_products_30d,avg_session_duration_sec_30d,sessions_cnt_7d,sessions_cnt_30d,days_since_last_purchase,orders_cnt_30d,orders_sum_usd_30d,orders_avg_usd_30d
38,43,2025-10-01,0,8,0,2,0.00,0.25,0.00,0.50,0,8,7560.0,0,1,29,1,350.88,350.88
40,46,2025-10-01,3,3,1,1,0.33,0.33,1.00,1.00,3,3,1980.0,1,1,0,1,26.37,26.37
45,51,2025-10-01,0,7,0,3,0.00,0.43,0.00,0.33,0,7,4920.0,0,1,12,1,60.27,60.27
52,61,2025-10-01,8,15,1,2,0.12,0.13,0.00,0.50,8,15,6720.0,1,2,24,1,16.09,16.09
68,80,2025-10-01,7,7,4,4,0.57,0.57,0.25,0.25,7,7,8700.0,1,1,5,1,253.02,253.02


'\nНеактивные пользователи (нет сессий за 30 дней):'

,customer_id,run_date,page_view_cnt_7d,page_view_cnt_30d,add_to_cart_cnt_7d,add_to_cart_cnt_30d,view_to_cart_conv_7d,view_to_cart_conv_30d,cart_to_purchase_conv_7d,cart_to_purchase_conv_30d,unique_products_7d,unique_products_30d,avg_session_duration_sec_30d,sessions_cnt_7d,sessions_cnt_30d,days_since_last_purchase,orders_cnt_30d,orders_sum_usd_30d,orders_avg_usd_30d
0,1,2025-10-01,0,0,0,0,0.0,0.0,0.0,0.0,0,0,0.0,0,0,97,0,0.0,0.0
1,2,2025-10-01,0,0,0,0,0.0,0.0,0.0,0.0,0,0,0.0,0,0,271,0,0.0,0.0
2,3,2025-10-01,0,0,0,0,0.0,0.0,0.0,0.0,0,0,0.0,0,0,-1,0,0.0,0.0
3,4,2025-10-01,0,0,0,0,0.0,0.0,0.0,0.0,0,0,0.0,0,0,-1,0,0.0,0.0
4,5,2025-10-01,0,0,0,0,0.0,0.0,0.0,0.0,0,0,0.0,0,0,243,0,0.0,0.0


'\nСводная статистика:'

,count,mean,std,min,25%,50%,75%,max
customer_id,17846.0,10014.60,5786.16,1.0,4993.25,10030.5,15019.75,19999.00
page_view_cnt_7d,17846.0,0.11,0.82,0.0,0.00,0.0,0.00,15.00
page_view_cnt_30d,17846.0,0.45,1.61,0.0,0.00,0.0,0.00,16.00
add_to_cart_cnt_7d,17846.0,0.03,0.28,0.0,0.00,0.0,0.00,6.00
add_to_cart_cnt_30d,17846.0,0.12,0.53,0.0,0.00,0.0,0.00,7.00
view_to_cart_conv_7d,17846.0,0.01,0.06,0.0,0.00,0.0,0.00,1.00
view_to_cart_conv_30d,17846.0,0.03,0.11,0.0,0.00,0.0,0.00,1.00
cart_to_purchase_conv_7d,17846.0,0.01,0.07,0.0,0.00,0.0,0.00,1.00
cart_to_purchase_conv_30d,17846.0,0.02,0.13,0.0,0.00,0.0,0.00,1.00
unique_products_7d,17846.0,0.11,0.82,0.0,0.00,0.0,0.00,15.00


1. **Активный пример (customer 43):** `days_since_last_purchase = 29` при `orders_cnt_30d = 1` — последняя покупка внутри 30-дневного окна, сходимость двух независимых признаков. `unique_products_30d = 8 = page_view_cnt_30d` — каждый просмотр был уникальным товаром, логично.
2. **Неактивные пользователи:** нули и` 0.0` везде, а `days_since_last_purchase` различается осмысленно:` 97/271/243` — покупал давно, но вне окна; `−1 `— не покупал никогда. Ровно по правилам ТЗ.
3. **Медиана days_since_last_purchase = −1:** больше половины клиентов (`17 846 − 8 230 = 9 616)` не имеют заказов — совпадает с уникальными покупателями из 1.1.
4. **Максимум days_since = 638:** это ровно дистанция от первого заказа в базе `(2024-01-01 07:08)` до` run_date 2025-10-01 00:00 UTC` в целых днях. Идеальная стыковка с границами дат из 1.1.
5. **Средние конверсии 0.01–0.03 при воронке 0.26/0.75 — не ошибка:** `~90%` строк это неактивные пользователи с нулями (деление на ноль → 0.0), они и тянут среднее вниз. У активных видим` 0.25–1.00`, как ожидалось.
6. **avg_session_duration_sec_30d max = 10 860 сек (≈3 часа)** — длинная, но реальная сессия; медиана `0`, т.к. большинство неактивны.

### 5.1 Артефакты для переноса в Python-модуль (фиксация логики)

1. **Чтение данных:** Загрузка таблиц (`customers`, `sessions`, `events`, `orders`) выполняется строго через SQL-запрос `SELECT * FROM public.{table}`. Параметры подключения не должны жестко прописываться в коде; в продуктивной среде используется безопасное подключение через `Airflow Connection.`
2. **Предобработка:** 
   * Все временные признаки (`signup_date`, `start_time`, `timestamp`, `order_time`) приводятся к единому часовому поясу (UTC). Строки с некорректными датами (значениями NaT) отбрасываются.
   * Выполняется дедупликация по первичному ключу (PK) для каждой таблицы.
   * Итоговый датасет `events` обогащается идентификаторами пользователей с помощью безопасного объединения `inner join` с таблицей `sessions` по полю `session_id`.
   * Для предотвращения утечки данных (data leakage) применяется строгое временное ограничение: все события должны быть строго раньше расчетной даты (`time < run_dt`, где `run_dt` — это `run_date` в формате `Timestamp` с таймзоной UTC).
3. **Окна агрегации:** Расчет признаков выполняется в рамках двух скользящих окон: за последние 7 дней (`[run_dt - 7d, run_dt]`) и за последние 30 дней (`[run_dt - 30d, run_dt]`).
4. **Правила расчета метрик и конверсий:**
   * Количество уникальных продуктов (`unique_products`) рассчитывается на основе таблицы `events` только для записей, где заполнено поле `product_id`.
   * Длина сессии определяется как разница между максимальным и минимальным временем событий внутри сессии (`max(timestamp) - min(timestamp)`). Если сессия не содержит событий, её длительность принимается равной 0.0. Средняя длительность сессий рассчитывается только на основе окна в 30 дней.
   * Количество сессий (`sessions_cnt`) считается по полю `start_time` в пределах заданных окон. Если у пользователя не было сессий, возвращается значение 0.
   * Количество дней с момента последней покупки (`days_since_last_purchase`) вычисляется как разница между расчетной датой и датой последнего заказа (`run_dt - max(order_time)`). Расчет производится по всей доступной истории. Если у пользователя нет заказов, признаку присваивается значение `-1`. Для всех остальных клиентов, попавших в выборку через `left join`, пропущенные значения заполняются нулями.
5. **Контракт данных:** Набор, порядок и состав колонок жестко фиксируются финальным списком `FEATURE_COLUMNS`, который в обязательном порядке включает в себя идентификатор клиента `customer_id` и расчетную дату `run_date`.
6. **Валидация и проверки:** Перед выгрузкой проверяется уникальность пар `(customer_id, run_date)`. Общее количество строк в итоговом датасете должно строго соответствовать числу уникальных клиентов. В данных не должно быть пропусков (NaN): для счетчиков за 7 и 30 дней, а также для признака `days_since_last_purchase` (если значение \(\geq 0\)) пропуски заполняются нулями. Конверсии должны находиться в диапазоне \(\geq 0\), а максимальное время событий во всех срезах должно быть строго меньше `run_dt`.
7. **Демонстрационный запуск:** Тестирование логики выполнено для расчетной даты `run_date = 2025-10-01`.

## Проверка: модуль и тетрадка идентичны

In [7]:
MODULE_PATH = "dags/calculate_batch_features.py"
spec = importlib.util.spec_from_file_location("calculate_batch_features", MODULE_PATH)
bf = importlib.util.module_from_spec(spec)
spec.loader.exec_module(bf)

# сырые таблицы customers/sessions/events/orders уже в памяти из задания 1.1,
# features — результат ячейки 2.2, RUN_DATE = "2025-10-01"
features_module = bf.build_batch_features(customers, sessions, events, orders, RUN_DATE)

pd.testing.assert_frame_equal(features_module.reset_index(drop=True),
                              features.reset_index(drop=True))
print("OK: модуль и тетрадка дают идентичный результат на", RUN_DATE)

OK: модуль и тетрадка дают идентичный результат на 2025-10-01


## Проверка резултатаов двух прогонов DAG

In [8]:
bucket = os.environ["S3_BUCKET"]
print("bucket:  ", bucket)

client = boto3.client(
    "s3",
    endpoint_url=os.environ["S3_ENDPOINT_URL"],
    aws_access_key_id=os.environ["S3_ACCESS_KEY"],
    aws_secret_access_key=os.environ["S3_SECRET_KEY"],
)
print("endpoint:", client.meta.endpoint_url)

# Фактические ключи срезов в бакете
resp = client.list_objects_v2(Bucket=bucket, Prefix="run_date=")
print("keys:", [o["Key"] for o in resp.get("Contents", [])])

# Сама сверка двух срезов
slices = {}
for rd in ["2025-09-01", "2025-10-01"]:
    obj = client.get_object(Bucket=bucket, Key=f"run_date={rd}/batch_features.csv")
    slices[rd] = pd.read_csv(obj["Body"])

a, b = slices["2025-09-01"], slices["2025-10-01"]
print("строк:", len(a), len(b))
print("набор колонок одинаковый:", list(a.columns) == list(b.columns))
print("customer_id одинаковые:  ", set(a["customer_id"]) == set(b["customer_id"]))
print("метка среза внутри:      ", a["run_date"].unique(), b["run_date"].unique())
print("значения различаются:    ",
      not a.drop(columns=["customer_id", "run_date"]).equals(b.drop(columns=["customer_id", "run_date"])))

bucket:   s3-ds-20260915-ff9d3b0716
endpoint: https://storage.yandexcloud.net
keys: ['run_date=2025-09-01/batch_features.csv', 'run_date=2025-10-01/batch_features.csv']
строк: 17846 17846
набор колонок одинаковый: True
customer_id одинаковые:   True
метка среза внутри:       ['2025-09-01'] ['2025-10-01']
значения различаются:     True


Все контрольные точки сошлись:
- `keys:` оба объекта на месте `(run_date=2025-09-01/..., run_date=2025-10-01/...)` — прогоны не перезаписали друг друга;
- строк: **17846 17846** — срезы полные, по всем клиентам;
- колонки и `customer_id` идентичны — инференсный срез посчитан на тех же пользователях и по тому же контракту признаков;
- метки среза внутри таблиц совпадают с датами прогонов;
- значения различаются — признаки пересчитаны под каждую дату, а не скопированы.